# Example 4: Hydrogen trapping in 1D

This notebook walks through [`eg_4.py`](eg_4.py). See [`README.md`](README.md) for the
full problem statement. This is the model at the heart of **FESTIM**.

## The problem

A **mobile** concentration `cm` diffuses through the rod and exchanges material with a
**trapped** concentration `ct` that is stuck in place. The two are **coupled** by a
reaction $R$:

$$ \frac{\partial c_m}{\partial t} - \nabla\cdot(D\,\nabla c_m) = -R, \qquad
   \frac{\partial c_t}{\partial t} = R, \qquad
   R = k\,c_m\,(n - c_t) - p\,c_t. $$

Material lost by the mobile field (`-R`) reappears in the trapped field (`+R`).

## What is new

Two fields that are genuinely **coupled** through the reaction term, a **mix of
element types** (continuous for the diffusing `cm`, discontinuous for the non-diffusing
`ct`), and a field (`ct`) with **no boundary condition**. The `cm * ct` product makes
the system **nonlinear**, which is why the Newton solver has mattered all along.

## 1. Backends and imports

The same backends as before (`mpi4py` for parallelism, `petsc4py` for solvers), plus
two new tools for a time-dependent run:

- **`tqdm`** for a progress bar over the time steps,
- **`dolfinx.io.VTXWriter`** to write the solution at each step to a `.bp` file you can
  open in ParaView.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
import basix
import numpy as np
import tqdm.autonotebook
from dolfinx import fem
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.mesh import create_mesh, locate_entities_boundary, meshtags
import ufl

from dolfinx.io import VTXWriter

## 2. The mesh

Unchanged from Example 1: a uniform 1D interval mesh on $(0, 1)$.

In [ ]:
# 100 points, 99 interval cells, on the domain (0, 1)
indices = np.linspace(0, 1, num=100)
gdim, shape, degree = 1, "interval", 1
domain = ufl.Mesh(basix.ufl.element("Lagrange", shape, degree, shape=(gdim,)))
mesh_points = np.reshape(indices, (len(indices), 1))
indexes = np.arange(mesh_points.shape[0])
cells = np.stack((indexes[:-1], indexes[1:]), axis=-1)
my_mesh = create_mesh(comm=MPI.COMM_WORLD, cells=cells, x=mesh_points, e=domain)
fdim = my_mesh.topology.dim - 1

## 3. A mixed space of two different elements

As in Example 3 we use a mixed element, but now the two fields use **different**
elements:

- `cm` uses a **continuous (CG)** element because it diffuses (it has a gradient term),
- `ct` uses a **discontinuous (DG)** element because it never appears under a gradient;
  it only changes pointwise through the reaction.

The split, sub-space and collapse helpers work exactly as before.

In [ ]:
element_CG = basix.ufl.element(
    basix.ElementFamily.P,
    my_mesh.basix_cell(),
    1,
    basix.LagrangeVariant.equispaced,
)
element_DG = basix.ufl.element(
    "DG",
    my_mesh.basix_cell(),
    1,
    basix.LagrangeVariant.equispaced,
)
elements = basix.ufl.mixed_element([element_CG, element_DG])

V = fem.functionspace(my_mesh, elements)
u = fem.Function(V)
u_n = fem.Function(V)

cm, ct = ufl.split(u)
cm_n, ct_n = ufl.split(u_n)
V1, V2 = V.sub(0), V.sub(1)
v_cm, v_ct = ufl.TestFunction(V)
cm_pp, ct_pp = u.sub(0).collapse(), u.sub(1).collapse()
_, map_cm_to_u = V.sub(0).collapse()
_, map_ct_to_u = V.sub(1).collapse()

## 4. Boundary conditions

Only the **mobile** field `cm` gets boundary conditions (`100` on the left, `0` on the
right). The trapped field `ct` has **none**: it is governed entirely by the reaction,
so its values are free to evolve at the boundary like everywhere else.

In [ ]:
num_facets = my_mesh.topology.index_map(fdim).size_local
mesh_facet_indices = np.arange(num_facets, dtype=np.int32)
tags_facets = np.full(num_facets, 0, dtype=np.int32)

entities_left = locate_entities_boundary(my_mesh, fdim, lambda x: np.isclose(x[0], 0))
entities_right = locate_entities_boundary(my_mesh, fdim, lambda x: np.isclose(x[0], 1))
tags_facets[entities_left] = 1
tags_facets[entities_right] = 2

facet_meshtags = meshtags(my_mesh, fdim, mesh_facet_indices, tags_facets)
my_mesh.topology.create_connectivity(fdim, my_mesh.topology.dim)

left_facets = facet_meshtags.find(1)
left_dofs_c1 = fem.locate_dofs_topological(V.sub(0), fdim, left_facets)
right_facets = facet_meshtags.find(2)
right_dofs_c1 = fem.locate_dofs_topological(V.sub(0), fdim, right_facets)

bc_left_cm = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(100)), left_dofs_c1, V1)
bc_right_cm = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(0)), right_dofs_c1, V1)
bcs = [bc_left_cm, bc_right_cm]

## 5. The coupled variational form

The reaction $R = k\,c_m(n - c_t) - p\,c_t$ is the heart of the model: mobile
hydrogen falling into empty traps minus trapped hydrogen released back.

We write `R` once, then put it into both equations with **opposite signs**: the mobile
field loses what the trapped field gains. The mobile equation also has a diffusion
term; the trapped equation does not.

In [ ]:
D = 0.1
dt = 0.05
k = 0.0001
p = 0.01
n = 1

reaction = -k * cm * (n - ct) + p * ct
F = ufl.dot(D * ufl.grad(cm), ufl.grad(v_cm)) * ufl.dx - reaction * v_cm * ufl.dx
F += ((cm - cm_n) / dt) * v_cm * ufl.dx

F += reaction * v_ct * ufl.dx
F += ((ct - ct_n) / dt) * v_ct * ufl.dx

## 6. The solver

Identical to the earlier examples: a `NonlinearProblem` driven by PETSc's SNES
(Newton) solver, configured with a direct LU solve (MUMPS) at each step. The trailing
loop clears the options out of the global PETSc database so they do not leak into
other solvers. See Example 1 for the line-by-line explanation.

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_stol": np.sqrt(np.finfo(dolfinx.default_real_type).eps) * 1e-2,
    "snes_atol": 1e-10,
    "snes_rtol": 1e-10,
    "snes_max_it": 30,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
solver = NonlinearProblem(
    F,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="festim_solver",
)

# clear the options out of the global PETSc database
snes = solver.solver
prefix = snes.getOptionsPrefix()
opts = PETSc.Options()
for key in petsc_options.keys():
    del opts[f"{prefix}{key}"]

## 7. Solve in a time loop

As before, with one output file per field. Watch the mobile profile diffuse in from
the left while the trapped field slowly fills behind it.

In [ ]:
writer1 = VTXWriter(MPI.COMM_WORLD, "trapping_cm.bp", cm_pp, "BP5")
writer2 = VTXWriter(MPI.COMM_WORLD, "trapping_ct.bp", ct_pp, "BP5")

final_time = 10
t = 0
progress = tqdm.autonotebook.tqdm(
    desc="Solving trapping problem", total=final_time, unit_scale=True
)
while t < final_time:
    solver.solve()
    u_n.x.array[:] = u.x.array

    cm_pp.x.array[:] = u.x.array[map_cm_to_u]
    ct_pp.x.array[:] = u.x.array[map_ct_to_u]

    writer1.write(t)
    writer2.write(t)

    t += dt
    progress.update(dt)

writer1.close()
writer2.close()

## Recap

You coupled two fields through a reaction term, mixed a continuous and a discontinuous
element in one space, and solved the resulting nonlinear system. This is the basic
FESTIM hydrogen-transport-with-trapping model. Open `trapping_cm.bp` and
`trapping_ct.bp` in [ParaView](https://www.paraview.org/) to see the mobile and trapped
profiles evolve.